# Consolidação dos resultados

Reúne numa tabela só todas as configurações avaliadas nos notebooks de `testes_treinamento`, por área
e sufixo, com MAE, RMSE e R² macro e a diferença de MAE contra o baseline.

Nada é treinado aqui. As métricas vêm dos CSVs já gravados; quando uma configuração só deixou as
previsões, e não as métricas, elas são recalculadas com a célula canônica de métricas, do mesmo jeito
que os notebooks de origem fazem.

## 1. Imports e caminhos

In [ ]:
# == caminhos do projeto ==
# A pasta deste caderno, dentro de notebooks/. Os CSVs sao lidos e gravados
# nela; as leituras entre pastas usam ../<outra pasta>. Para que esses caminhos
# valham qualquer que seja o diretorio em que o kernel foi aberto, o bloco
# abaixo localiza a raiz do repositorio -- a pasta que contem notebooks/ e
# documents/ -- e entra na pasta do caderno. No Colab, onde nao existe
# repositorio e os arquivos ficam soltos ao lado do caderno, nada e alterado.
import os

PASTA_CADERNO = "colabs_S4/testes_extras/testes_treinamento"


def raiz_repositorio(inicio=None):
    """Sobe a arvore de diretorios ate a raiz do repositorio; None se nao achar."""
    atual = os.path.abspath(inicio or os.getcwd())
    while True:
        if all(os.path.isdir(os.path.join(atual, p)) for p in ("notebooks", "documents")):
            return atual
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai


RAIZ_REPO = raiz_repositorio()
if RAIZ_REPO:
    os.chdir(os.path.join(RAIZ_REPO, "notebooks", PASTA_CADERNO))
print("Diretorio de trabalho:", os.path.abspath("."))

In [ ]:
import os, json

import pandas as pd
import numpy as np

PASTA_DADOS = "colabs_S3"

BASES = [
    ".", "..", os.path.join("..", ".."), os.path.join("..", "..", ".."), "/content",
    "notebooks", os.path.join("..", "notebooks"),
    os.path.join("notebooks", PASTA_DADOS),
]

def resolve(pasta, arquivo):
    tentativas = []
    # A pasta declarada tem prioridade sobre o arquivo solto: sem isso, um
    # homonimo numa pasta vizinha pode ser lido no lugar do arquivo pedido.
    for partes in ((pasta, arquivo), (arquivo,)):
        for base in BASES:
            caminho = os.path.join(base, *partes)
            tentativas.append(caminho)
            if os.path.exists(caminho):
                return caminho
    raise FileNotFoundError(
        "Arquivo nao encontrado: " + arquivo
        + ". Diretorio atual: " + os.path.abspath(".")
        + ". Caminhos tentados: " + "; ".join(tentativas)
    )

AREAS = ["coil_temps", "ct_knobs", "ipc_comp", "ipc_cooling",
         "main_vent_sys", "mainback", "oven_hmi"]
SUFIXOS = ["_perfil", "_13cat"]
CHAVE = ["sufixo", "area"]

print("Diretorio:", os.path.abspath("."))

## 2. Catálogo de configurações

Cada configuração avaliada, com o notebook que a produziu e o arquivo de onde vêm as métricas.

Três configurações aparecem como `previsoes` e não como `metricas`: `mesclagem_rf_gb.ipynb`,
`varredura_pesos.ipynb` e `validacao_peso_otimo.ipynb` não gravam CSV algum, apenas relatórios em
`documents/extras`. Para essas, as métricas são recalculadas aqui a partir de `previsoes_rf_gb.csv`.

A coluna `simplicidade` é a ordem de desempate usada adiante: primeiro os modelos individuais
padrão, depois os individuais com os hiperparâmetros do artefato, depois a mesclagem de peso fixo e
por último as mesclagens cujo peso foi escolhido na própria validação. Dentro do último grupo, o peso
global vem antes do peso por área, por ser um único valor por sufixo em vez de um por área.

O assert interrompe a execução se algum arquivo de origem faltar, em vez de reexecutar notebooks.

In [ ]:
# Cada configuracao avaliada nos notebooks desta pasta, com a origem das metricas.
# "metricas" = o CSV ja traz MAE, RMSE e R2 macro; "previsoes" = precisa ser calculado aqui.
CONFIGURACOES = [
    {"configuracao": "rf_padrao", "algoritmo": "rf", "simplicidade": 1,
     "notebook": "baseline_metricas_por_area.ipynb",
     "origem": "baseline_metricas_por_area.csv", "tipo": "metricas"},
    {"configuracao": "gb_padrao", "algoritmo": "gb", "simplicidade": 2,
     "notebook": "baseline_metricas_por_area.ipynb",
     "origem": "baseline_metricas_por_area.csv", "tipo": "metricas"},
    {"configuracao": "rf_artefato", "algoritmo": "rf", "simplicidade": 3,
     "notebook": "treino_rf_gb_otimizado.ipynb",
     "origem": "comparacao_otimizado_baseline.csv", "tipo": "metricas"},
    {"configuracao": "gb_artefato", "algoritmo": "gb", "simplicidade": 4,
     "notebook": "treino_rf_gb_otimizado.ipynb",
     "origem": "comparacao_otimizado_baseline.csv", "tipo": "metricas"},
    {"configuracao": "mescla_peso_0,5", "algoritmo": "rf+gb", "simplicidade": 5,
     "notebook": "mesclagem_rf_gb.ipynb",
     "origem": "previsoes_rf_gb.csv", "tipo": "previsoes"},
    {"configuracao": "mescla_peso_global", "algoritmo": "rf+gb", "simplicidade": 6,
     "notebook": "validacao_peso_otimo.ipynb",
     "origem": "previsoes_rf_gb.csv", "tipo": "previsoes"},
    {"configuracao": "mescla_peso_otimo_area", "algoritmo": "rf+gb", "simplicidade": 7,
     "notebook": "varredura_pesos.ipynb",
     "origem": "previsoes_rf_gb.csv", "tipo": "previsoes"},
    {"configuracao": "mescla_peso_otimo_artefato", "algoritmo": "rf+gb", "simplicidade": 8,
     "notebook": "mesclagem_otimizada.ipynb",
     "origem": "comparacao_mesclagem_otimizada.csv", "tipo": "metricas"},
]
catalogo = pd.DataFrame(CONFIGURACOES)

# Todo arquivo de origem precisa existir localmente; nenhum notebook e reexecutado.
faltando = []
for arquivo in sorted(set(catalogo["origem"])) + ["limites_ativos.csv"]:
    try:
        resolve(PASTA_DADOS, arquivo)
    except FileNotFoundError:
        faltando.append(arquivo)
assert not faltando, ("arquivos de origem ausentes: " + ", ".join(faltando)
                      + ". Rode os notebooks que os produzem antes desta consolidacao.")

print(f"{len(catalogo)} configuracoes | {len(set(catalogo['origem']))} arquivos de origem, todos presentes")
catalogo[["configuracao", "algoritmo", "notebook", "origem", "tipo", "simplicidade"]]

## 3. Métricas

Célula canônica de `colabs_S3/preparacao_testes/metricas.ipynb` (seção 1), copiada sem alteração.

In [ ]:
import os
import json

import numpy as np
import pandas as pd


MARGEM_KPI = 0.10   # margem de erro acordada no TAPI


def _valida(y_real, y_previsto):
    """Converte para vetores e recusa entradas que produziriam metrica sem sentido.

    A conversao para numpy e necessaria porque a subtracao entre duas Series do pandas
    alinha pelos indices, e nao pela posicao. Uma Series com indice original e outra
    com indice reiniciado produziriam diferencas entre pares errados, em silencio.

    O alvo precisa ser estritamente positivo. A restricao e legitima neste projeto,
    pois trata-se de temperatura absoluta em graus Celsius, cujo minimo na base e de
    27,69 graus. Ela existe para transformar em erro visivel o engano de passar valores
    padronizados, caso em que o alvo se aproximaria de zero e o erro percentual
    explodiria em vez de acusar o problema.
    """
    y_real = np.asarray(y_real, dtype=float).ravel()
    y_previsto = np.asarray(y_previsto, dtype=float).ravel()

    if y_real.shape != y_previsto.shape:
        raise ValueError(f"dimensoes diferentes: {y_real.shape} e {y_previsto.shape}")
    if y_real.size == 0:
        raise ValueError("nenhuma observacao a avaliar")
    if np.isnan(y_real).any():
        raise ValueError("ha valores ausentes em y_real; a mascara deve ser aplicada antes")
    if np.isnan(y_previsto).any():
        raise ValueError("ha valores ausentes em y_previsto")
    if (y_real <= 0).any():
        raise ValueError(
            "y_real contem valor nao positivo; o erro percentual exige temperatura "
            "absoluta em graus Celsius, e nao valores padronizados")

    return y_real, y_previsto


def avaliar(y_real, y_previsto, limite=None):
    """Metricas de erro de um componente, com o alvo em graus Celsius.

    Sempre devolvidas:

    n            numero de observacoes que sustentam as metricas
    MAE          erro absoluto medio, na unidade do alvo
    RMSE         raiz do erro quadratico medio, na mesma unidade
    R2           fracao da variancia do alvo explicada pelo modelo
    vies         erro medio com sinal, positivo quando o modelo superestima
    RMSE/MAE     razao entre as duas, indicando concentracao dos erros
    MAPE_%       erro percentual absoluto medio
    WAPE_%       erro absoluto total dividido pela soma dos alvos
    dentro_10_%  proporcao de predicoes dentro da margem de 10 por cento do TAPI

    Devolvidas apenas quando um limite de seguranca e informado:

    limite_C          o proprio limite, para conferencia
    n_reprova_real    ensaios em que o componente de fato atinge o limite
    acerto_decisao_%  concordancia entre a decisao real e a prevista
    falso_aprovado    ensaios reprovados na realidade e aprovados pelo modelo
    falso_reprovado   ensaios aprovados na realidade e reprovados pelo modelo

    O R2 sai ausente quando o alvo assume um unico valor, situacao em que ele e
    indefinido e em que a implementacao do scikit-learn devolve zero, valor que se
    confunde com um resultado legitimo.
    """
    y_real, y_previsto = _valida(y_real, y_previsto)

    erro = y_previsto - y_real
    mae = float(np.mean(np.abs(erro)))

    # A raiz e calculada explicitamente: o parametro squared do mean_squared_error foi
    # removido do scikit-learn na versao 1.9, e o root_mean_squared_error so existe a
    # partir da 1.4. O calculo direto funciona em qualquer versao.
    rmse = float(np.sqrt(np.mean(erro ** 2)))

    if np.unique(y_real).size < 2:
        r2 = float("nan")
    else:
        ss_res = float(np.sum(erro ** 2))
        ss_tot = float(np.sum((y_real - np.mean(y_real)) ** 2))
        r2 = 1.0 - ss_res / ss_tot

    relativo = np.abs(erro) / y_real

    resultado = {
        "n": int(y_real.size),
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "vies": float(np.mean(erro)),
        "RMSE/MAE": (rmse / mae) if mae > 0 else float("nan"),
        "MAPE_%": 100.0 * float(np.mean(relativo)),
        "WAPE_%": 100.0 * float(np.sum(np.abs(erro)) / np.sum(y_real)),
        "dentro_10_%": 100.0 * float(np.mean(relativo <= MARGEM_KPI)),
    }

    if limite is not None and np.isfinite(limite):
        reprova_real = y_real >= limite
        reprova_prevista = y_previsto >= limite
        resultado.update({
            "limite_C": float(limite),
            "n_reprova_real": int(reprova_real.sum()),
            "acerto_decisao_%": 100.0 * float(np.mean(reprova_real == reprova_prevista)),
            "falso_aprovado": int(np.sum(reprova_real & ~reprova_prevista)),
            "falso_reprovado": int(np.sum(~reprova_real & reprova_prevista)),
        })

    return resultado


def consolidar(tabela, grupo="dataset",
               metricas=("MAE", "RMSE", "R2", "vies", "MAPE_%", "WAPE_%", "dentro_10_%")):
    """Consolida metricas por componente em metricas por area.

    A macro-media atribui o mesmo peso a cada componente e responde se o modelo funciona
    bem em todas as partes do produto. A micro-media pondera pelo numero de observacoes
    e responde qual o erro tipico de uma leitura qualquer.

    O R2 nao e comparavel entre areas, pois seu denominador e a variancia do alvo de
    cada area. Ele indica se o modelo supera a media dentro da propria area.
    """
    if grupo not in tabela.columns:
        raise ValueError(f"coluna de agrupamento ausente: {grupo}")
    if "n" not in tabela.columns:
        raise ValueError("a tabela precisa da coluna n para a micro-media")

    metricas = [m for m in metricas if m in tabela.columns]
    linhas = []

    for nome, bloco in tabela.groupby(grupo):
        linha = {grupo: nome, "componentes": len(bloco), "n_total": int(bloco["n"].sum())}
        for m in metricas:
            valores = bloco[m]
            validos = valores.notna()
            linha[f"{m}_macro"] = (float(valores[validos].mean())
                                   if validos.any() else float("nan"))
            if validos.any() and bloco.loc[validos, "n"].sum() > 0:
                linha[f"{m}_micro"] = float(
                    np.average(valores[validos], weights=bloco.loc[validos, "n"]))
            else:
                linha[f"{m}_micro"] = float("nan")

        # As metricas de decisao sao contagens, e por isso somadas em vez de promediadas
        for contagem in ("n_reprova_real", "falso_aprovado", "falso_reprovado"):
            if contagem in bloco.columns:
                linha[contagem] = int(bloco[contagem].fillna(0).sum())
        linhas.append(linha)

    return pd.DataFrame(linhas).sort_values(grupo).reset_index(drop=True)


print("Metricas definidas: avaliar, consolidar")

## 4. Leitura dos resultados gravados

In [ ]:
previsoes = pd.read_csv(resolve(PASTA_DADOS, "previsoes_rf_gb.csv"))
limites = pd.read_csv(resolve(PASTA_DADOS, "limites_ativos.csv")).set_index("tc_name")
baseline = pd.read_csv(resolve(PASTA_DADOS, "baseline_metricas_por_area.csv"))
otimizado = pd.read_csv(resolve(PASTA_DADOS, "comparacao_otimizado_baseline.csv"))
mescla_art = pd.read_csv(resolve(PASTA_DADOS, "comparacao_mesclagem_otimizada.csv"))

print("previsoes_rf_gb:", previsoes.shape, "| baseline:", baseline.shape,
      "| otimizado:", otimizado.shape, "| mescla artefato:", mescla_art.shape)

## 5. Recálculo das mesclagens sem CSV

Mesmo método dos notebooks de origem: mesclagem por média ponderada componente a componente, com o
peso do rf indo de 0 a 1 em passos de 0,05, e `avaliar` recebendo o limite de segurança.

O peso ótimo por área sai do menor MAE macro por sufixo e área. O peso global sai do menor MAE macro
por sufixo, com as sete áreas consolidadas juntas, como em `validacao_peso_otimo.ipynb`.

In [ ]:
def mesclar(rf, gb, peso=0.5):
    """Combinacao ponderada das previsoes; peso e o do rf e (1 - peso) o do gb."""
    if not 0 <= peso <= 1:
        raise ValueError(f"peso fora de [0, 1]: {peso}")
    rf = np.asarray(rf, dtype=float)
    gb = np.asarray(gb, dtype=float)
    if rf.shape != gb.shape:
        raise ValueError(f"dimensoes diferentes: {rf.shape} e {gb.shape}")
    return peso * rf + (1 - peso) * gb


def limite_de(col):
    lim = limites["limite_num"].get(col)
    return float(lim) if lim is not None and pd.notna(lim) else None


PESOS = np.round(np.linspace(0, 1, 21), 2)

linhas = []
for (sufixo, area, col), p in previsoes.groupby(["sufixo", "area", "componente"], sort=False):
    limite = limite_de(col)
    for peso in PESOS:
        linhas.append({"sufixo": sufixo, "area": area, "componente": col, "peso": peso,
                       **avaliar(p["real"], mesclar(p["rf"], p["gb"], peso), limite=limite)})
varrido = pd.DataFrame(linhas)

# por area, para o peso 0,5 e para o peso otimo de cada area
por_area = pd.concat(
    [consolidar(b, grupo="area").assign(sufixo=s, peso=w)
     for (s, w), b in varrido.groupby(["sufixo", "peso"], sort=False)], ignore_index=True)

# por sufixo, sobre os 74 componentes das sete areas juntas, para o peso global
por_sufixo = pd.concat(
    [consolidar(b, grupo="sufixo").assign(peso=w) for w, b in varrido.groupby("peso", sort=False)],
    ignore_index=True)
peso_global = (por_sufixo.loc[por_sufixo.groupby("sufixo")["MAE_macro"].idxmin(), ["sufixo", "peso"]]
               .set_index("sufixo")["peso"])

print(f"{len(PESOS)} pesos varridos | peso global por sufixo: {peso_global.to_dict()}")

## 6. Montagem por configuração

Cada configuração vira um bloco com as mesmas colunas. RF e GB entram com peso 1 e 0, que é o que
eles são na escala da mesclagem, de modo que a coluna de peso fique preenchida em todas as linhas.

In [ ]:
MACRO = ["MAE_macro", "RMSE_macro", "R2_macro"]
SAIDA = ["MAE", "RMSE", "R2"]


def de_por_area(seletor):
    """Extrai MAE, RMSE e R2 macro de por_area segundo uma funcao (sufixo, area) -> peso."""
    linhas = []
    for sufixo in SUFIXOS:
        for area in AREAS:
            w = seletor(sufixo, area)
            r = por_area[(por_area["sufixo"] == sufixo) & (por_area["area"] == area)
                         & np.isclose(por_area["peso"], w)]
            assert len(r) == 1, f"{sufixo}/{area}/peso {w}: {len(r)} linhas"
            r = r.iloc[0]
            linhas.append({"sufixo": sufixo, "area": area, "peso": w,
                           **{s: float(r[m]) for s, m in zip(SAIDA, MACRO)}})
    return pd.DataFrame(linhas)


# peso otimo por area: menor MAE macro na varredura; em empate fica o menor peso
otimo_area = (por_area.loc[por_area.groupby(CHAVE)["MAE_macro"].idxmin(), CHAVE + ["peso"]]
              .set_index(CHAVE)["peso"])

blocos = {}
for modelo, nome in (("rf", "rf_padrao"), ("gb", "gb_padrao")):
    b = baseline[baseline["modelo"] == modelo][CHAVE + MACRO].copy()
    b.columns = CHAVE + SAIDA
    blocos[nome] = b.assign(peso=np.where(modelo == "rf", 1.0, 0.0))

for modelo, nome in (("rf", "rf_artefato"), ("gb", "gb_artefato")):
    b = otimizado[otimizado["modelo"] == modelo][
        CHAVE + [f"{m.replace('_macro', '')}_macro_otimizado" for m in MACRO]].copy()
    b.columns = CHAVE + SAIDA
    blocos[nome] = b.assign(peso=np.where(modelo == "rf", 1.0, 0.0))

blocos["mescla_peso_0,5"] = de_por_area(lambda s, a: 0.5)
blocos["mescla_peso_global"] = de_por_area(lambda s, a: float(peso_global[s]))
blocos["mescla_peso_otimo_area"] = de_por_area(lambda s, a: float(otimo_area[(s, a)]))

b = mescla_art[CHAVE + ["peso_otimo", "MAE_mescla", "RMSE_mescla", "R2_mescla"]].copy()
b.columns = CHAVE + ["peso"] + SAIDA
blocos["mescla_peso_otimo_artefato"] = b[CHAVE + SAIDA + ["peso"]]

print("blocos montados:", {k: len(v) for k, v in blocos.items()})

## 7. Tabela consolidada

O Δ de MAE é contra o menor MAE entre RF e GB do baseline, na mesma área e sufixo. Negativo quando a
configuração erra menos que o melhor modelo padrão daquela área.

In [ ]:
tabela = pd.concat(
    [v.assign(configuracao=k) for k, v in blocos.items()], ignore_index=True)
tabela = tabela.merge(catalogo[["configuracao", "algoritmo", "simplicidade"]], on="configuracao")

# Delta contra o menor MAE entre RF e GB do baseline, na mesma area e sufixo.
referencia = (baseline.pivot(index=CHAVE, columns="modelo", values="MAE_macro")[["rf", "gb"]]
              .min(axis=1).rename("MAE_referencia"))
tabela = tabela.join(referencia, on=CHAVE)
tabela["delta_MAE"] = tabela["MAE"] - tabela["MAE_referencia"]

tabela = (tabela[["sufixo", "area", "configuracao", "algoritmo", "peso",
                  "MAE", "RMSE", "R2", "MAE_referencia", "delta_MAE", "simplicidade"]]
          .sort_values(["sufixo", "area", "simplicidade"])
          .reset_index(drop=True))

assert tabela.notna().all().all(), f"ha celulas vazias:\n{tabela.isna().sum()}"
presenca = tabela.groupby(["sufixo", "configuracao"])["area"].nunique()
assert set(tabela["sufixo"]) == set(SUFIXOS), f"sufixos: {sorted(set(tabela['sufixo']))}"
assert (presenca == 7).all(), f"area faltando:\n{presenca[presenca != 7]}"
assert len(tabela) == len(CONFIGURACOES) * len(SUFIXOS) * len(AREAS), f"{len(tabela)} linhas"

print(f"{len(tabela)} linhas = {len(CONFIGURACOES)} configuracoes x "
      f"{len(SUFIXOS)} sufixos x {len(AREAS)} areas | sem celulas vazias")
tabela.head(8).round(3)

## 8. Melhor configuração por área e sufixo

A linha de menor MAE é marcada. Em empate exato, abaixo de 1e-9, fica a configuração mais simples
pela ordem do catálogo.

In [ ]:
TOL = 1e-9

def marca(bloco):
    menor = bloco["MAE"].min()
    empatados = bloco[bloco["MAE"] - menor < TOL]
    # em empate exato fica a configuracao de menor indice de simplicidade
    vencedor = empatados["simplicidade"].idxmin()
    fora = pd.Series(False, index=bloco.index)
    fora.loc[vencedor] = True
    return fora

tabela["melhor"] = (tabela.groupby(CHAVE, group_keys=False)
                    .apply(marca, include_groups=False).astype(bool))

assert tabela.groupby(CHAVE)["melhor"].sum().eq(1).all(), "deveria haver uma melhor por area e sufixo"

contagem = (tabela[tabela["melhor"]]
            .groupby("configuracao").size()
            .reindex(catalogo["configuracao"]).fillna(0).astype(int)
            .rename("vezes_melhor"))
print("vezes marcada como melhor:")
print(contagem.to_string())

## 9. Gravação

In [ ]:
pasta = os.path.dirname(resolve(PASTA_DADOS, "limites_ativos.csv"))
caminho = os.path.join(pasta, "consolidacao_resultados.csv")
tabela.to_csv(caminho, index=False)
print(caminho)
print(f"{tabela.shape[0]} linhas, {tabela.shape[1]} colunas")

## 10. Tabelas em Markdown

Uma tabela por sufixo, com três casas decimais e vírgula decimal. A configuração marcada como melhor
da área aparece em negrito.

In [ ]:
def num(v, sinal=False):
    if pd.isna(v):
        return "–"
    # Um delta que arredonda a zero na precisao publicada sai sem sinal: modelos
    # identicos diferem por ruido de ponto flutuante, e "-0,000" sugeriria direcao.
    if sinal and abs(round(v, 3)) < 5e-4:
        return "0,000"
    texto = f"{v:+.3f}" if sinal else f"{v:.3f}"
    return texto.replace(".", ",")


COLS = ["Área", "Configuração", "Algoritmo", "MAE", "RMSE", "R2", "Δ MAE"]

for sufixo in SUFIXOS:
    bloco = tabela[tabela["sufixo"] == sufixo]
    print(f"### {sufixo}\n")
    print("| " + " | ".join(COLS) + " |")
    print("| " + " | ".join(["---"] * 3 + ["---:"] * 4) + " |")
    for _, r in bloco.iterrows():
        nome = f"**{r['configuracao']}**" if r["melhor"] else r["configuracao"]
        print("| " + " | ".join([r["area"], nome, r["algoritmo"],
                                 num(r["MAE"]), num(r["RMSE"]), num(r["R2"]),
                                 num(r["delta_MAE"], sinal=True)]) + " |")
    print()